In [4]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from load_doc import documents
from dotenv import load_dotenv
load_dotenv()
    
model_name = "Snowflake/snowflake-arctic-embed-l-v2.0"
# model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = HuggingFaceEmbeddings(model_name=model_name)


def custom_relevance_score_fn(distance: float) -> float:
    return distance

vectore_db=FAISS.from_documents(
    documents=documents,
    embedding=embedder,
    distance_strategy="MAX_INNER_PRODUCT",
    relevance_score_fn=custom_relevance_score_fn,
)

def search_filter(metadata):
    return metadata["topic"]=="Human Resources"

retriever = vectore_db.as_retriever(
    # search_type: similarity/similarity_score_threshold/mmr 
    search_type = "similarity_score_threshold",
    search_kwargs = {
        "k": 5,
        "score_threshold": 0.6,
        "fetch_k": 20,
        # "filter": search_filter
    }
)


def docs_to_context(docs):
    context = "\n\n".join(f"{doc.page_content}\n{doc.metadata['answer']}" for doc in docs)
    print(docs)
    print('--------')
    print(context)
    print('--------')
    return {
        "docs": docs,
        "context": context
    }


from langchain_core.prompts import PromptTemplate

prompt_template = """\
You are a helpful company internal assistant.
Answer the question using ONLY the context below.
You can paraphrase and infer when wording is different but meaning is the same.
If the context truly does not provide enough information, say "I don't know".

Context:
{context}

Question:
{question}
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template
)

import os
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough

# DeepSeek is OpenAI-compatible — just point to a different base_url
llm = ChatOpenAI(
    model="gpt-4o-mini",
    openai_api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,
)

query = "How many days of vacation can we get each year?"
documents = retriever.invoke(query)

rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": retriever | docs_to_context
    } | prompt | llm
)

query = "How many days of vacation can we get each year?"

resp = rag_chain.invoke(query)
print(resp.content)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6022.43it/s]


[Document(id='0b6a9187-4212-46f3-922f-b49f938860f9', metadata={'source': 'D:\\Projects\\2026bigdream\\bootcamp\\course\\rag\\practice\\data\\common.jsonl', 'seq_num': 1, 'topic': 'Human Resources', 'answer': 'Annual leave entitlement depends on company policy and local labor laws. Typically, employees get 10 to 20 days per year.'}, page_content='How many annual leave days am I entitled to?'), Document(id='055f6a1a-81db-4fb0-8e91-b3df56efde80', metadata={'source': 'D:\\Projects\\2026bigdream\\bootcamp\\course\\rag\\practice\\data\\common.jsonl', 'seq_num': 19, 'topic': 'Finance', 'answer': 'In the finance department, vacation days are tracked based on payroll and accrual records. The system calculates leave balances according to financial accounting rules, and unused leave may be carried forward or compensated based on budget policies.'}, page_content='How many vacation days are employees allowed according to finance records?')]
--------
How many annual leave days am I entitled to?
Annu

In [5]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

retriever_BM25 = BM25Retriever.from_documents(documents, k=4)
ensemble_retriever = EnsembleRetriever(
    retrievers=[retriever, retriever_BM25], weights=[0.5, 0.5]
)
rag_chain3 = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=ensemble_retriever,
    return_source_documents=True,
)

rag_chain3.invoke('How should I handle a customer complaint?')

{'query': 'How should I handle a customer complaint?',
 'result': "To handle a customer complaint effectively, you should:\n\n1. **Listen Actively**: Allow the customer to express their concerns without interruption. Show empathy and understanding.\n2. **Acknowledge the Issue**: Validate their feelings and acknowledge the problem they are facing.\n3. **Apologize**: Offer a sincere apology for the inconvenience caused, even if it wasn't your fault.\n4. **Ask Questions**: Gather more information to fully understand the issue and the customer's perspective.\n5. **Provide Solutions**: Offer possible solutions or alternatives to resolve the complaint. Involve the customer in the decision-making process if appropriate.\n6. **Take Action**: Implement the agreed-upon solution promptly.\n7. **Follow Up**: After resolving the issue, follow up with the customer to ensure they are satisfied with the resolution and to reinforce that their feedback is valued.\n\nBy following these steps, you can eff

In [ ]:
for chunk in rag_chain.stream('How should I handle a customer complaint?'):
    print(chunk.content, end="", flush=True)
print()

[Document(id='e69da54a-6bf0-438e-bbd1-2b5096bc4b30', metadata={'source': 'D:\\Projects\\2026bigdream\\bootcamp\\course\\rag\\practice\\data\\common.jsonl', 'seq_num': 9, 'topic': 'Customer Service', 'answer': 'Acknowledge the issue, offer a prompt solution, and ensure customer satisfaction to build long-term trust.'}, page_content='What is the best way to handle customer complaints?'), Document(id='d3df31d6-6b66-4d45-acb9-76947833653d', metadata={'source': 'D:\\Projects\\2026bigdream\\bootcamp\\course\\rag\\practice\\data\\common.jsonl', 'seq_num': 11, 'topic': 'Customer Service', 'answer': 'Apologize for the inconvenience, ask for specifics, and work to resolve the issue, or escalate it to the relevant department.'}, page_content='What should I do if a customer is dissatisfied with the service?'), Document(id='84e6f682-b3dc-421a-bf32-684a842bb1fd', metadata={'source': 'D:\\Projects\\2026bigdream\\bootcamp\\course\\rag\\practice\\data\\common.jsonl', 'seq_num': 16, 'topic': 'Customer S